# Experiments Analysis

Retrieve and filter LangSmith evaluation runs for flexible post-hoc analysis.

In [28]:
import sys
import os
import json

sys.path.insert(0, os.path.abspath(".."))

from typing import Any, Callable, Dict, List, Optional

from langsmith import Client

client = Client()

In [44]:
def load_experiment_runs(
    experiment_prefix: str = "",
    dataset_name: str = "Dataset v1.",
) -> List[Dict[str, Any]]:
    """
    Retrieve all evaluation runs from LangSmith experiments and return them
    as a flat list of dicts ready for filtering and inspection.

    Each record contains:
        experiment         — experiment (project) name
        task_id            — the eval task identifier
        thread_id          — the pipeline checkpoint thread id
        run_id             — LangSmith run id
        scores             — dict of {score_key: float} from all evaluators
        <score_key>        — every score also flattened to top level for easy filtering
        final_code         — generated code from the pipeline state
        retrieval_context  — RAG context from the pipeline state
        comments           — dict of {feedback_key: comment_string} from evaluators
        statements         — parsed JSON from first available comment (or None if no valid JSON)
        outputs            — raw LangSmith run outputs (full pre_computed_state inside)

    Args:
        experiment_prefix: Only include experiments whose name starts with this string.
                           Empty string means include all experiments for the dataset.
        dataset_name:      LangSmith dataset name used to scope experiments.
    """
    records: List[Dict[str, Any]] = []

    projects = list(client.list_projects(reference_dataset_name=dataset_name))
    if experiment_prefix:
        projects = [p for p in projects if p.name.startswith(experiment_prefix)]

    print(f"Found {len(projects)} experiment(s) matching prefix '{experiment_prefix}'")

    for project in projects:
        runs = list(client.list_runs(project_name=project.name, is_root=True))

        if not runs:
            continue

        # Bulk-fetch all feedback for this project in one call
        run_ids = [str(r.id) for r in runs]
        feedback_by_run: Dict[str, Dict[str, Any]] = {rid: {} for rid in run_ids}
        for fb in client.list_feedback(run_ids=run_ids):
            rid = str(fb.run_id)

            if fb.score is not None:
                feedback_by_run[rid][fb.key] = fb.score
            
            # Store the comment for each feedback key as well
            if fb.comment:
                if f"{fb.key}_comment" not in feedback_by_run[rid]:
                    feedback_by_run[rid][f"{fb.key}_comment"] = fb.comment

        for run in runs:
            rid = str(run.id)
            feedback_data = feedback_by_run.get(rid, {})
            scores = {k: v for k, v in feedback_data.items() if not k.endswith("_comment")}
            outputs = run.outputs or {}
            pre_state = outputs.get("pre_computed_state") or {}

            task_id = (
                str(outputs.get("task_id") or "").strip()
                or str(pre_state.get("task_id") or "").strip()
            )

            # Extract and parse statements from feedback comments
            # Try to parse as JSON; if it fails, keep the raw comment
            statements_parsed = None
            comments_dict = {}
            
            for key, value in feedback_data.items():
                if key.endswith("_comment"):
                    comment_key = key.replace("_comment", "")
                    
                    # Try to parse as JSON
                    if value:
                        try:
                            parsed = json.loads(value)
                            comments_dict[comment_key] = parsed
                            if statements_parsed is None:
                                statements_parsed = parsed
                        except (json.JSONDecodeError, ValueError):
                            # If JSON parsing fails, keep raw comment string
                            comments_dict[comment_key] = value
                    else:
                        comments_dict[comment_key] = value

            record: Dict[str, Any] = {
                "experiment": project.name,
                "task_id": task_id,
                "thread_id": outputs.get("thread_id", ""),
                "run_id": rid,
                "scores": scores,
                "final_code": pre_state.get("final_code") or pre_state.get("generated_code") or "",
                "retrieval_context": pre_state.get("retrieval_context", ""),
                "comments": comments_dict,
                "statements": statements_parsed,
                "outputs": outputs,
            }
            # Flatten scores to top level for convenient filtering
            record.update(scores)

            records.append(record)

    print(f"Loaded {len(records)} total run(s)")
    return records

In [30]:
def filter_runs(
    records: List[Dict[str, Any]],
    **conditions: Any,
) -> List[Dict[str, Any]]:
    """
    Filter records by field values.

    Each keyword argument is a field name. The value can be:
    - A scalar     → exact equality match  (e.g., code_execution_score=0.0)
    - A callable   → predicate on the field (e.g., task_id=lambda v: "add" in v)

    Missing fields evaluate to None against the condition.

    Example:
        filter_runs(records, code_statements_score=1.0, code_execution_score=0.0)
        filter_runs(records, task_id=lambda v: v.startswith("add"))
    """
    result = []
    for record in records:
        if all(
            (cond(record.get(key)) if callable(cond) else record.get(key) == cond)
            for key, cond in conditions.items()
        ):
            result.append(record)
    return result


def display_runs(
    records: List[Dict[str, Any]],
    show_code: bool = False,
    show_context: bool = False,
    code_preview_chars: int = 600,
    context_preview_chars: int = 400,
) -> None:
    """
    Pretty-print a list of run records.

    Args:
        records:              Output of load_experiment_runs / filter_runs.
        show_code:            Print a preview of final_code.
        show_context:         Print a preview of retrieval_context.
        code_preview_chars:   Max chars to show for code preview.
        context_preview_chars: Max chars to show for context preview.
    """
    SEP = "─" * 70
    print(f"{len(records)} record(s)\n{SEP}")

    for i, r in enumerate(records, 1):
        scores = r.get("scores", {})
        score_str = "  ".join(
            f"{k}={v:.2f}" for k, v in sorted(scores.items()) if v is not None
        )
        print(f"\n[{i}]  task_id={r.get('task_id', '?')!r}")
        print(f"     experiment: {r.get('experiment', '?')}")
        print(f"     thread_id:  {r.get('thread_id', '?')}")
        print(f"     scores:     {score_str or '(none)'}")

        if show_code:
            code = r.get("final_code", "") or ""
            snippet = code[:code_preview_chars]
            ellipsis = "..." if len(code) > code_preview_chars else ""
            print(f"     final_code:\n{snippet}{ellipsis}")

        if show_context:
            ctx = r.get("retrieval_context", "") or ""
            snippet = ctx[:context_preview_chars]
            ellipsis = "..." if len(ctx) > context_preview_chars else ""
            print(f"     retrieval_context:\n{snippet}{ellipsis}")

    print(f"\n{SEP}")

## Usage Examples

In [45]:
# --- Load ---
# Empty prefix loads all experiments for the dataset.
# Supply a prefix string to scope to a specific experiment family.
EXPERIMENT_PREFIX = "NO RAG"       # e.g. "2) With concision"
DATASET_NAME = "Dataset v1."

records = load_experiment_runs(
    experiment_prefix=EXPERIMENT_PREFIX,
    dataset_name=DATASET_NAME,
)

Found 8 experiment(s) matching prefix 'NO RAG'
Loaded 48 total run(s)


In [47]:

# --- Verify JSON parsing is integrated into comments ---
print("=== Verifying integrated JSON parsing ===\n")

r = records[0]
print(f"Task: {r['task_id']}\n")

for comment_type, comment_value in r['comments'].items():
    print(f"[{comment_type}]")
    print(f"  Type: {type(comment_value).__name__}")
    
    if isinstance(comment_value, dict):
        print(f"  Dict keys: {list(comment_value.keys())}")
        if 'pass' in comment_value:
            print(f"  Pass: {comment_value['pass']}")
    elif isinstance(comment_value, list):
        print(f"  List length: {len(comment_value)}")
        if comment_value and isinstance(comment_value[0], dict):
            print(f"  First item keys: {list(comment_value[0].keys())}")
    else:
        print(f"  Raw string (parsing failed): {comment_value[:100]}")
    print()

# Count how many comments are parsed vs raw
parsed_count = sum(1 for r in records for c in r['comments'].values() if not isinstance(c, str))
raw_count = sum(1 for r in records for c in r['comments'].values() if isinstance(c, str))
total_comments = sum(len(r['comments']) for r in records)

print(f"Summary:")
print(f"  Total comments: {total_comments}")
print(f"  Parsed (JSON): {parsed_count}")
print(f"  Raw (string): {raw_count}")


=== Verifying integrated JSON parsing ===

Task: add_task

[code_execution_score]
  Type: dict
  Dict keys: ['pass', 'output', 'errors']
  Pass: False

[awrapper]
  Type: str
  Raw string (parsing failed): ValueError('Failed to extract JSON from response. Content preview: ```json\n[\n  {\n    "statement":

[rag_statements_score]
  Type: list
  List length: 6
  First item keys: ['statement', 'status', 'evidence', 'reasoning']

Summary:
  Total comments: 144
  Parsed (JSON): 138
  Raw (string): 6


In [6]:
# --- Filter: code statements correct but code didn't execute ---
interesting = filter_runs(
    records,
    #code_statements_score=lambda x: x != 1,
    code_execution_score=0.0,
)
len(interesting)

8

In [7]:
from collections import Counter

challenging_cases = Counter(r["task_id"] for r in interesting)
challenging_cases

Counter({'add_task': 4,
         'update_task_status_synthetic': 3,
         'retrieve_tasks': 1})

In [8]:
challenging_cases.keys()

dict_keys(['add_task', 'update_task_status_synthetic', 'retrieve_tasks'])

In [9]:
interesting = [r for r in interesting if r["task_id"] == "add_task"]

In [10]:
display_runs(
    interesting, 
    show_code=True, 
    show_context=False, 
    code_preview_chars=20000,
    context_preview_chars=20000
)

4 record(s)
──────────────────────────────────────────────────────────────────────

[1]  task_id='add_task'
     experiment: NO RAG: 2) no rag (default).-62974b28
     thread_id:  add_task_20260312110531
     scores:     code_execution_score=0.00  rag_statements_score=0.83
     final_code:
import os
import dotenv
import requests
import datetime
import sys

dotenv.load_dotenv()

def add_notion_task(task_title: str, task_date: str, task_importance: int, project_id: str) -> dict:
    """
    Adds a new task to the Notion Tasks database.
    """
    notion_token = os.getenv('NOTION_TOKEN')
    if not notion_token:
        raise ValueError("NOTION_TOKEN environment variable is not set.")

    url = "https://api.notion.com/v1/pages"
    headers = {
        "Authorization": f"Bearer {notion_token}",
        "Content-Type": "application/json",
        "Notion-Version": "2022-06-28"
    }

    payload = {
        "parent": {"database_id": project_id},
        "properties": {
            "Name":

In [16]:
# --- Mismatch cases: code statements < 1 but execution == 1 ---
# Find cases where code has issues but still executed successfully

mismatch_cases = filter_runs(
    records,
    code_statements_score=lambda x: x is not None and x < 1,
    code_execution_score=1.0,
)

print(f"Found {len(mismatch_cases)} total mismatch cases\n")

# Group by task_id and display samples
from collections import defaultdict
by_task = defaultdict(list)
for r in mismatch_cases:
    by_task[r["task_id"]].append(r)

for task_id in sorted(by_task.keys()):
    samples = by_task[task_id]  # Get up to 2 samples
    print(f"\n{'='*70}")
    print(f"TASK: {task_id} ({len(samples)} samples shown)")
    print(f"{'='*70}")
    display_runs(
        samples,
        show_code=True,
        show_context=False,
        code_preview_chars=20000,
    )

Found 20 total mismatch cases


TASK: add_task (1 samples shown)
1 record(s)
──────────────────────────────────────────────────────────────────────

[1]  task_id='add_task'
     experiment: NO RAG: no rag & tasks data source IN RETRIEVAL CONTEXT.-b052b68a
     thread_id:  add_task_20260311235847
     scores:     code_execution_score=1.00  code_statements_score=0.83  rag_statements_score=0.67
     final_code:
import os
import dotenv
import requests
import datetime
import sys

dotenv.load_dotenv()

def add_task(title: str, date: str, importance: int, project_id: str, token: str = os.getenv('NOTION_TOKEN'), db_id: str = os.getenv('NOTION_TASKS_DATABASE_ID')) -> dict:
    """
    Adds a new task to the Notion Tasks database.
    
    :param title: The name of the task.
    :param date: The date in YYYY-MM-DD format.
    :param importance: The importance level (1-4).
    :param project_id: The ID of the related project.
    :param token: Notion API token.
    :param db_id: Notion database I